### Data Sourcing

In [ ]:
from __future__ import annotations

import csv
import re
from pathlib import Path

import pandas as pd

# Define input and output folders
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODIFIED_DIR = ROOT / "data" / "modified" / "ira"
OUT_DIR = ROOT / "data" / "processed" / "ira_wide_cleaned"
PROCESSED_DIR = ROOT / "data" / "processed"
IRA_FILES = sorted(MODIFIED_DIR.glob("*.csv"))

# Parsing rules for PSY IRA tables
YEAR_RE = re.compile(r"\b(19|20)\d{2}\b")
STOP_PREFIXES = ("notes:", "source:")
ENCODINGS = ("utf-8-sig", "cp1252", "latin-1")
AGGREGATE_PROVINCES = {"philippines", "national capital region", "ncr"}
NULL_TOKENS = {"", "-", "\u2013", "\u2014", "na", "n/a", "nan", "null", "\ufffd"}
SOURCE_YEAR_LIMITS = {"psy_prov_ira_2000_2002": {"2000", "2001", "2002"}}


### Dataframe Reshaping

In [ ]:
def clean_header(value: str) -> str:
    match = YEAR_RE.search(value or "")
    return match.group(0) if match else ""


def clean_label(value: str) -> str:
    value = re.sub(r"\s+", " ", (value or "").strip())
    value = re.sub(r"\s+[0-9]+$", "", value)
    value = re.sub(r"\s+[a-z]$", "", value)
    return value.strip()


def clean_amount(value: str) -> object:
    value = (value or "").strip()
    normalized = re.sub(r"\s+", "", value).lower()
    return pd.NA if normalized in NULL_TOKENS else value.replace(",", "").strip()


def read_csv_rows(path: Path) -> list[list[str]]:
    for encoding in ENCODINGS:
        try:
            with path.open(newline="", encoding=encoding) as handle:
                return list(csv.reader(handle))
        except UnicodeDecodeError:
            pass
    raise UnicodeDecodeError(path.name, b"", 0, 1, "Unsupported file encoding")


def has_years(row: list[str]) -> bool:
    return sum(bool(clean_header(cell)) for cell in row) >= 2


def process_ira_file(path: Path) -> pd.DataFrame:
    rows = read_csv_rows(path)
    header_idx = next(i for i, row in enumerate(rows) if has_years(row))
    header = rows[header_idx]

    year_cols = [(i, clean_header(cell)) for i, cell in enumerate(header) if clean_header(cell)]
    if path.stem in SOURCE_YEAR_LIMITS:
        year_cols = [(i, year) for i, year in year_cols if year in SOURCE_YEAR_LIMITS[path.stem]]

    data_rows = []
    seen_header = False
    current_region = ""
    pending_region = ""

    for row in rows[header_idx + 1:]:
        row += [""] * (len(header) - len(row))
        first = (row[0] if row else "").strip().lower()

        if all(not cell.strip() for cell in row):
            continue
        if first.startswith(STOP_PREFIXES):
            break
        if has_years(row):
            seen_header = True
            continue
        if seen_header and first.startswith("table"):
            continue

        raw_region = row[0] if row else ""
        raw_province = row[1] if len(row) > 1 else ""
        values = [clean_amount(row[i] if i < len(row) else "") for i, _ in year_cols]

        if raw_region.strip().lower().startswith(("table", "continued")):
            continue
        if all(pd.isna(value) for value in values):
            if raw_region and not raw_region.startswith(" ") and not raw_province.strip():
                pending_region = clean_label(raw_region)
            continue

        if raw_region.startswith(" ") and not raw_province.strip():
            row_label = clean_label(raw_region)
            if pending_region and row_label.lower().startswith("in "):
                current_region = clean_label(f"{pending_region} {row_label}")
                region, province = current_region, ""
            else:
                current_region = pending_region or current_region
                region, province = current_region, row_label
            pending_region = ""
        else:
            region = clean_label(raw_region)
            province = clean_label(raw_province)
            pending_region = ""
            current_region = region if region and not province else current_region

        if region and province and province.lower() not in AGGREGATE_PROVINCES:
            data_rows.append([region, province, *values])

    df = pd.DataFrame(data_rows, columns=["reg_name", "prov_name", *[year for _, year in year_cols]])
    df[df.columns[2:]] = df[df.columns[2:]].apply(pd.to_numeric, errors="coerce")
    return df.sort_values("prov_name", key=lambda col: col.str.casefold()).reset_index(drop=True)


# Apply parser to all raw IRA files
ira_wide = {path.stem: process_ira_file(path) for path in IRA_FILES}

# Validate cleaned wide outputs and summarize missing values
null_summary = []
for name, df in ira_wide.items():
    year_cols = df.columns[2:]
    assert list(df.columns[:2]) == ["reg_name", "prov_name"]
    assert df["reg_name"].ne("").all()
    assert df["prov_name"].ne("").all()
    assert not df["prov_name"].str.lower().isin(AGGREGATE_PROVINCES).any()
    assert df[year_cols].notna().any(axis=1).all()

    null_summary.append({
        "source_file": name,
        "rows": len(df),
        "year_columns": len(year_cols),
        "missing_values": int(df[year_cols].isna().sum().sum()),
    })

pd.DataFrame(null_summary)


### Time-Series Formatting

In [ ]:
# Convert cleaned wide tables into one long panel
ira_long = pd.concat(
    [
        df.melt(
            id_vars=["reg_name", "prov_name"],
            var_name="year",
            value_name="ira_funding_million_php",
        ).assign(source_file=name)
        for name, df in ira_wide.items()
    ],
    ignore_index=True,
)

# Enforce modeling-friendly data types
ira_long["year"] = ira_long["year"].astype(int)
ira_long["ira_funding_million_php"] = pd.to_numeric(
    ira_long["ira_funding_million_php"],
    errors="coerce",
)

# Sort and protect against repeated PSY overlap years
ira_long = (
    ira_long.sort_values(["prov_name", "year", "source_file"])
    .drop_duplicates(["prov_name", "year"], keep="last")
    .sort_values(["reg_name", "prov_name", "year"])
    .reset_index(drop=True)
)

ira_long = ira_long[["reg_name", "prov_name", "year", "ira_funding_million_php", "source_file"]]

# Final long-panel checks
assert not ira_long.duplicated(["prov_name", "year"]).any()
assert pd.api.types.is_integer_dtype(ira_long["year"])
assert pd.api.types.is_numeric_dtype(ira_long["ira_funding_million_php"])


### IRA Dataframe Exporting

In [ ]:
# Export cleaned wide files and master long panel
OUT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for name, df in ira_wide.items():
    df.to_csv(OUT_DIR / f"{name}.csv", index=False)

output_path = PROCESSED_DIR / "ira_funding_master_long.csv"
ira_long.to_csv(output_path, index=False)

# Quick sanity check output
print(f"Wrote {output_path.relative_to(ROOT)}")
print(f"Successfully processed and saved {len(ira_long)} rows.")
print(f"Year range: {ira_long['year'].min()}-{ira_long['year'].max()}")
print("Missing IRA values:", int(ira_long["ira_funding_million_php"].isna().sum()))
display(ira_long.head())
